# Craft My Book

Uses the `llm` package in `src/` (OpenAI provider) to draft book content.

In [1]:
import sys
from pathlib import Path


sys.path.insert(0, str(Path.cwd() / "src"))

from llm import get_client, chat
import config

In [ ]:
# Generic demo of the raw `llm` package - not tied to any pipeline component/config,
# just a standalone example of calling a model directly.
client = get_client("openai")
model = "gpt-4o-mini"

In [3]:
prompt = "Write an engaging opening paragraph for a book about ..."

response = chat(client, model, prompt)
print(response)

Certainly! What specific theme or subject would you like the opening paragraph to focus on?


## Module 1.2 — Speech Processing (Whisper)

Turns a lecture recording — video or audio — into a structured, timestamped,
domain-aware transcript (`src/ingestion/speech.py`, `src/ingestion/vocab.py`).
Requires the `ffmpeg` binary on PATH and `faster-whisper` installed
(`pip install -r requirements.txt`).

**Design**: `ingestion.base.Ingestor` is an abstract strategy — `ingest(source_path) ->
IngestedDocument` — that every source type implements (speech today; PDF/DOCX/PPTX/
image strategies for Pipeline A land the same way later). `SpeechIngestor` is the
speech strategy; `ingestion.registry.get_ingestor_class()` picks the right strategy
for a file by extension, so code that walks a folder of mixed sources never branches
on file type itself. `IngestedDocument`/`IngestedSegment` are the common output shape
every strategy produces, regardless of what ran underneath.

**Config**: `src/config/config.yaml` is nested by component, not one global
provider/model. `ingestion.speech` alone has three independently tunable model
choices — `whisper` (transcription), `cleaning_llm` (post-transcription cleanup),
`vocab_llm` (domain vocab extraction) — so e.g. cleaning can run on a cheap model
while vocab extraction runs on something else, without either affecting the other
or any future component (toc/writer). `src/config/__init__.py` loads this into typed
dataclasses; everything reads from `config.INGESTION.speech.*`, nothing is hardcoded
in a function signature. Whisper defaults to `"small"` for local iteration — bump
`ingestion.speech.whisper.model_size` to `"large-v3"` in the YAML for real
lecture-quality transcription once you have a stable connection (and ideally a GPU).

Flow: `extract_vocab` (LLM pass over slide titles/filenames/headings — vocab is
per-corpus, never hardcoded) → `extract_audio` (ffmpeg; handles video or audio
sources) → `transcribe_audio` (faster-whisper, vocab-primed, VAD-filtered, word
timestamps, no cross-segment conditioning) → `clean_transcript` (conservative LLM
pass — fixes terms/punctuation, changes nothing else) → `IngestedDocument` JSON.

In [ ]:
from ingestion import extract_audio, extract_vocab, vocab_material_from_filenames, bootstrap_vocab_from_audio

# Each sub-component gets its own client/model, independently tunable via
# config.INGESTION.speech.* - vocab extraction doesn't have to run on the same
# provider/model as transcript cleaning. base_url is only used when provider: custom.
_vocab_cfg = config.INGESTION.speech.vocab_llm
vocab_client = get_client(_vocab_cfg.provider, base_url=_vocab_cfg.base_url)
vocab_model = _vocab_cfg.model

# data/harvard-speech.wav: ~34s of real spoken English (public-domain Harvard sentences
# test corpus) - use this to smoke-test the pipeline locally before pointing it at a
# real lecture.
source_path = "data/harvard-speech.wav"

# Preferred: derive vocab from material that already exists around the recording
# (slide titles / filenames here; could also be PDF headings).
sibling_files: list[str] = []  # no slides for this sample -> falls through to bootstrap
vocab_material = vocab_material_from_filenames(sibling_files)
vocab = extract_vocab(vocab_material, vocab_client, vocab_model)

# Fallback: audio is the ONLY source (no slides/useful filenames) - bootstrap vocab from
# a fast draft transcription pass instead of skipping priming altogether.
if not vocab:
    audio_path = extract_audio(source_path)
    vocab = bootstrap_vocab_from_audio(audio_path, vocab_client, vocab_model)

vocab

In [ ]:
# End to end via the strategy: registry picks the Ingestor for this file, ingest()
# runs source -> audio -> transcribe -> clean, save_document() writes the JSON. Once
# PDF/DOCX/image strategies exist, code that walks a mixed folder still looks like this.
from ingestion import get_ingestor_class, save_document

_cleaning_cfg = config.INGESTION.speech.cleaning_llm
cleaning_client = get_client(_cleaning_cfg.provider, base_url=_cleaning_cfg.base_url)
cleaning_model = _cleaning_cfg.model

IngestorClass = get_ingestor_class(source_path)  # -> SpeechIngestor, by extension
ingestor = IngestorClass(client=cleaning_client, clean_model=cleaning_model, vocab=vocab)

document = ingestor.ingest(source_path)
out_path = save_document(document, config.INGESTION.output_dir)

print(f"{document.source_type}: {len(document.segments)} segments, {document.metadata['duration']:.0f}s -> {out_path}")
document.segments[0]

## Module 1.3 — Layout Parsing (MinerU)

A PDF is a visual layout, not a logical document — positioned text spans with no
inherent concept of "heading" vs "paragraph" vs "caption", and no guarantee reading
order matches storage order (two-column papers). `LayoutIngestor`
(`src/ingestion/layout/mineru.py`) shells out to the `mineru` CLI (`pip install
"mineru[core]"`) to reconstruct that structure deterministically — no LLM call in this
stage at all. Same `Ingestor` interface as `SpeechIngestor`, so `get_ingestor_class()`
dispatches to it automatically for `.pdf`/`.docx`/`.pptx`/`.xlsx`.

First run downloads model weights (layout/OCR/table/formula detection) — see
`MINERU_MODEL_SOURCE` in `.env` if the default mirror is slow, or pre-fetch with
`mineru-models-download`. Config: `ingestion.layout.backend` / `.work_dir` in
`config.yaml`.

Each block becomes an `IngestedSegment` — `block_type` (text/heading/image/table/
equation/chart/...), `page`, `bbox`, and type-specific fields (`img_path`,
`table_body`, captions, ...) all preserved in `metadata`. Images/tables/equations
carry an `img_path` but MinerU doesn't describe them — that's Module 1.4.

In [ ]:
# data/attention-is-all-you-need.pdf: real paper (headings, paragraphs, figures,
# tables, equations) - good smoke test since it exercises every MinerU block type.
from ingestion import get_ingestor_class, save_document

pdf_path = "data/attention-is-all-you-need.pdf"

LayoutIngestorClass = get_ingestor_class(pdf_path)  # -> LayoutIngestor, by extension
layout_ingestor = LayoutIngestorClass()  # no LLM client needed - purely deterministic

layout_doc = layout_ingestor.ingest(pdf_path)
out_path = save_document(layout_doc, config.INGESTION.output_dir)

print(f"{layout_doc.source_type}: {len(layout_doc.segments)} blocks, {layout_doc.metadata['page_count']} pages -> {out_path}")
layout_doc.segments[1]  # the title heading, typically

## Module 1.4 — Figure Understanding

MinerU *finds* every image/chart/table/equation; this stage *understands* what's
inside them (`src/ingestion/vision/describe.py`). Not an `Ingestor` — it's an
enrichment pass over the `IngestedDocument` Module 1.3 already produced.

No model/provider is hardcoded anywhere in this module — every call goes through the
generic `llm` package, same as every other component, so the vision model is entirely
a `config.yaml` edit (`ingestion.vision.vlm`: any hosted vision-capable API, or
`provider: custom` + `base_url` for a self-hosted OpenAI-compatible endpoint serving
e.g. Qwen2-VL). Defaults to `gpt-4o-mini` since it's already configured and
vision-capable.

Two things straight from the brief, not obvious defaults:
- Visuals are never described in isolation — each call gets the surrounding page text
  plus every other visual on that same page (`ingestion.vision.batch_by: page`), so a
  figure can be described with awareness of a table right next to it.
- The description **replaces** `segment.text` (so Pipeline B can treat it like any
  other paragraph), but the raw MinerU extraction (LaTeX/HTML) and the image path are
  kept in `segment.metadata["raw_text"]` / `["img_path"]` — never deleted.

This is the first call that actually sends images to a model — real cost/latency per
run (one call per page that has a visual, not one call per image).

In [ ]:
from ingestion import describe_visuals

_vlm_cfg = config.INGESTION.vision.vlm
vlm_client = get_client(_vlm_cfg.provider, base_url=_vlm_cfg.base_url)

described_doc = describe_visuals(layout_doc, vlm_client, _vlm_cfg.model)

visuals = [s for s in described_doc.segments if s.metadata.get("description")]
print(f"described {len(visuals)} visual blocks")
visuals[0]